# Hands-On Lab: Linear Shift-Invariant Systems

**Estimated time:** 60-75 minutes  
**Tools:** NumPy and Matplotlib

An LSI system has two properties:

1. **Linearity:** $T(a x_1 + b x_2)=aT(x_1)+bT(x_2)$
2. **Shift invariance:** if the input shifts, the output shifts by the same amount without changing shape.

For an LSI system, the output is convolution with the impulse response $h$:

$$y[n]=x[n]*h[n]=\sum_k x[k]h[n-k].$$

Run the cells in order. Attempt every **Your turn** cell before opening or running its solution.

## Learning outcomes

By the end of the lab, you should be able to:

- interpret an impulse response;
- calculate a discrete convolution;
- test linearity numerically;
- test shift invariance without circular-shift mistakes;
- explain why convolution filters are LSI and median filters are not;
- apply the same ideas to a small 2D image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

def T(x, h):
    """LSI system: full discrete convolution of x with h."""
    return np.convolve(x, h, mode='full')

def shift_right(x, amount):
    """Delay a finite signal using zero padding, not wraparound."""
    return np.concatenate([np.zeros(amount), np.asarray(x)])

def stem(ax, signal, title, color='C0'):
    n = np.arange(len(signal))
    markerline, stemlines, baseline = ax.stem(n, signal, basefmt='k-')
    plt.setp(markerline, color=color)
    plt.setp(stemlines, color=color)
    ax.set_title(title)
    ax.set_xlabel('n')
    ax.grid(alpha=0.3)

print('Setup complete.')

## Problem 1: Find the impulse response

Consider the system

$$y[n]=0.25x[n]+0.50x[n-1]+0.25x[n-2].$$

1. Create a unit impulse $\delta[n]$.
2. Pass it through the system.
3. Predict the output before running the code.

The output produced by an impulse is called the **impulse response**.

In [ ]:
# YOUR TURN: replace None with suitable arrays.
impulse = None
h = None

# Uncomment after completing the arrays.
# response = T(impulse, h)
# print('Impulse response:', response)

### Solution 1

In [ ]:
impulse = np.array([1., 0., 0., 0., 0.])
h = np.array([0.25, 0.50, 0.25])
response = T(impulse, h)

fig, axes = plt.subplots(2, 1, figsize=(8, 5))
stem(axes[0], impulse, 'Input: unit impulse')
stem(axes[1], response, 'Output: impulse response', 'C3')
plt.tight_layout()
plt.show()

print('Impulse response:', response)
assert np.allclose(response[:len(h)], h)
print('Check passed: the beginning of the output equals h.')

## Problem 2: Calculate convolution by hand

Let

$$x=[1,2,1], \qquad h=[1,-1].$$

Calculate the full convolution $y=x*h$ by hand. The output must contain

$$N+M-1=3+2-1=4$$

values. Enter your prediction below before using `np.convolve`.

In [ ]:
x = np.array([1., 2., 1.])
h_difference = np.array([1., -1.])

# YOUR TURN: replace the question marks with four numbers.
# predicted_y = np.array([?, ?, ?, ?])
# print('Your prediction:', predicted_y)

### Solution 2

Each sample of $x$ produces a shifted and scaled copy of $h$.

In [ ]:
predicted_y = np.array([1., 1., -1., -1.])
computed_y = T(x, h_difference)

contributions = []
for position, value in enumerate(x):
    contribution = np.zeros(len(computed_y))
    contribution[position:position + len(h_difference)] = value * h_difference
    contributions.append(contribution)

fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
for i, contribution in enumerate(contributions):
    stem(axes[i], contribution, f'Contribution from x[{i}] = {x[i]:g}')
stem(axes[-1], computed_y, 'Sum of contributions: y = x * h', 'C3')
plt.tight_layout()
plt.show()

print('Predicted:', predicted_y)
print('Computed: ', computed_y)
assert np.allclose(predicted_y, computed_y)
print('Check passed.')

## Problem 3: Test linearity

A system is linear when

$$T(ax_1+bx_2)=aT(x_1)+bT(x_2).$$

Use $a=2$ and $b=-0.5$. Compute both sides and compare them.

In [ ]:
x1 = np.array([0., 1., 2., 0., 0.])
x2 = np.array([1., 0., 0., 1., 0.])
h = np.array([0.25, 0.50, 0.25])
a, b = 2.0, -0.5

# YOUR TURN
# left = ...
# right = ...
# error = ...
# print('Maximum error:', error)

### Solution 3

In [ ]:
left = T(a * x1 + b * x2, h)
right = a * T(x1, h) + b * T(x2, h)
error = np.max(np.abs(left - right))

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
stem(axes[0], left, 'Left side: T(a x1 + b x2)')
stem(axes[1], right, 'Right side: aT(x1) + bT(x2)', 'C3')
plt.tight_layout()
plt.show()

print('Maximum error:', error)
print('Linear:', np.allclose(left, right))
assert np.allclose(left, right)

## Problem 4: Test shift invariance

Shift the input right by three positions. Compare:

1. filtering the shifted input;
2. shifting the original output.

Use zero padding. `np.roll` performs a circular shift and is not appropriate for this test.

In [ ]:
x = np.array([0., 1., 2., 1., 0.])
h = np.array([0.25, 0.50, 0.25])
delay = 3

# YOUR TURN
# shifted_x = ...
# output_from_shifted_input = ...
# shifted_original_output = ...

### Solution 4

In [ ]:
shifted_x = shift_right(x, delay)
output_from_shifted_input = T(shifted_x, h)
shifted_original_output = shift_right(T(x, h), delay)

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
stem(axes[0], shifted_x, f'Input shifted right by {delay}')
stem(axes[1], output_from_shifted_input, 'T(shifted input)', 'C1')
stem(axes[2], shifted_original_output, 'Shifted T(original input)', 'C3')
plt.tight_layout()
plt.show()

print('Shift invariant:',
      np.allclose(output_from_shifted_input, shifted_original_output))
assert np.allclose(output_from_shifted_input, shifted_original_output)

## Problem 5: Which systems are LSI?

Classify each system as LSI or not LSI. Give a reason.

1. $T_1(x[n])=3x[n]$
2. $T_2(x[n])=x[n]^2$
3. $T_3(x[n])=\frac{x[n-1]+x[n]+x[n+1]}{3}$
4. $T_4(x[n])=n\,x[n]$
5. A three-sample median filter

### Solution 5

1. **LSI.** Multiplication by a constant is linear and does not depend on location.
2. **Not LSI.** Squaring is nonlinear; generally $(x_1+x_2)^2\neq x_1^2+x_2^2$.
3. **LSI.** It is convolution with the fixed kernel $[1/3,1/3,1/3]$.
4. **Not LSI.** The multiplier depends on $n$, so shifting the input changes the operation applied to it.
5. **Not LSI.** A median filter is shift invariant under a consistent boundary rule, but it is not linear. Both properties are required.

## Problem 6: Demonstrate that the median is nonlinear

Use the two neighborhoods below. Test whether

$$M(A+B)=M(A)+M(B).$$

In [ ]:
A = np.array([[1, 1, 1],
              [1, 1, 2],
              [2, 2, 2]])

B = np.array([[0, 0, 0],
              [0, 1, 0],
              [0, 0, 0]])

# YOUR TURN
# left = np.median(...)
# right = np.median(...) + np.median(...)
# print(left, right)

### Solution 6

In [ ]:
left = np.median(A + B)
right = np.median(A) + np.median(B)

print('M(A + B) =', left)
print('M(A) + M(B) =', right)
print('Linear:', np.isclose(left, right))

assert left == 2
assert right == 1

## Problem 7: Extend the idea to a 2D image

An image impulse is an image containing one bright pixel. For a 2D LSI system, filtering this image reveals the 2D impulse response.

Run the code and explain why the output looks like the kernel centered at the impulse position.

In [ ]:
def correlate2d_zero(image, kernel):
    """Small educational 2D cross-correlation with zero padding."""
    image = np.asarray(image, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode='constant')
    output = np.zeros_like(image, dtype=float)

    for row in range(image.shape[0]):
        for col in range(image.shape[1]):
            patch = padded[row:row + kh, col:col + kw]
            output[row, col] = np.sum(patch * kernel)
    return output

image_impulse = np.zeros((11, 11))
image_impulse[5, 5] = 1

kernel = np.array([[1, 2, 1],
                   [2, 4, 2],
                   [1, 2, 1]], dtype=float)
kernel /= kernel.sum()

image_response = correlate2d_zero(image_impulse, kernel)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# All panels use the same 11 x 11 coordinate system.
full_extent = (-0.5, 10.5, 10.5, -0.5)
kernel_extent = (3.5, 6.5, 6.5, 3.5)

shown0 = axes[0].imshow(
    image_impulse, cmap='gray', interpolation='nearest',
    extent=full_extent
)
shown1 = axes[1].imshow(
    kernel, cmap='gray', interpolation='nearest',
    extent=kernel_extent
)
shown2 = axes[2].imshow(
    image_response, cmap='gray', interpolation='nearest',
    extent=full_extent
)

titles = [
    '2D impulse (11 x 11)',
    'Kernel at its 3 x 3 size',
    'Filter response (11 x 11)'
]

for ax, title in zip(axes, titles):
    ax.set_xlim(-0.5, 10.5)
    ax.set_ylim(10.5, -0.5)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

for ax, shown in zip(axes, [shown0, shown1, shown2]):
    fig.colorbar(shown, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

### Solution 7

The impulse contains one nonzero pixel with value 1. Every zero-valued pixel contributes nothing. The one nonzero pixel contributes one shifted copy of the kernel, so the output is the kernel located at the impulse position.

The helper above computes **cross-correlation**, which is the convention used by many image-processing libraries. Cross-correlation with a fixed kernel is also linear and shift invariant. For a symmetric kernel such as this one, cross-correlation and convolution produce the same output.

## Challenge: Verify 2D linearity

Create two small images, `I1` and `I2`. Verify numerically that

$$T(2I_1-I_2)=2T(I_1)-T(I_2).$$

In [ ]:
# YOUR TURN
# I1 = ...
# I2 = ...
# left_2d = ...
# right_2d = ...

### Challenge solution

In [ ]:
I1 = np.zeros((11, 11))
I2 = np.zeros((11, 11))
I1[3, 3] = 1
I2[7, 6] = 1

combined_input = 2 * I1 - I2
left_2d = correlate2d_zero(combined_input, kernel)
right_2d = (
    2 * correlate2d_zero(I1, kernel)
    - correlate2d_zero(I2, kernel)
)

difference = np.abs(left_2d - right_2d)

print('I1 =\n', I1)
print('\nI2 =\n', I2)
print('\nCombined input 2*I1 - I2 =\n', combined_input)
print('\nLeft side T(2*I1 - I2) =\n', left_2d)
print('\nRight side 2*T(I1) - T(I2) =\n', right_2d)
print('\nAbsolute difference =\n', difference)
print('\nMaximum error:', difference.max())
print('Linear:', np.allclose(left_2d, right_2d))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

input_plots = [
    (I1, 'Input I1'),
    (I2, 'Input I2'),
    (combined_input, 'Combined input: 2I1 - I2')
]

for ax, (data, title) in zip(axes[0], input_plots):
    limit = max(1.0, np.abs(data).max())
    shown = ax.imshow(
        data, cmap='coolwarm', vmin=-limit, vmax=limit,
        interpolation='nearest'
    )
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(shown, ax=ax, fraction=0.046)

response_limit = max(
    np.abs(left_2d).max(),
    np.abs(right_2d).max()
)

shown_left = axes[1, 0].imshow(
    left_2d, cmap='coolwarm',
    vmin=-response_limit, vmax=response_limit,
    interpolation='nearest'
)
axes[1, 0].set_title('Left: T(2I1 - I2)')

shown_right = axes[1, 1].imshow(
    right_2d, cmap='coolwarm',
    vmin=-response_limit, vmax=response_limit,
    interpolation='nearest'
)
axes[1, 1].set_title('Right: 2T(I1) - T(I2)')

shown_difference = axes[1, 2].imshow(
    difference, cmap='magma', interpolation='nearest'
)
axes[1, 2].set_title('Absolute difference')

for ax, shown in zip(
    axes[1],
    [shown_left, shown_right, shown_difference]
):
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(shown, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

assert np.allclose(left_2d, right_2d)

## Exit questions

1. What is an impulse response?
2. Why does knowing $h$ completely describe an LSI system?
3. What two numerical comparisons test linearity?
4. Why should `np.roll` be used carefully when testing ordinary shift invariance?
5. Is a median filter shift invariant? Is it linear? Is it LSI?
6. Why are convolution and cross-correlation identical for a symmetric kernel?

### Short answers

1. The output of a system when its input is a unit impulse.
2. Any input is a sum of shifted and scaled impulses; linearity and shift invariance make the output the corresponding sum of shifted and scaled copies of $h$.
3. Compare $T(ax_1+bx_2)$ with $aT(x_1)+bT(x_2)$.
4. `np.roll` wraps values from one boundary to the other, creating a circular shift.
5. With a consistent boundary rule it is shift invariant, but it is nonlinear, so it is not LSI.
6. Flipping a symmetric kernel does not change it.